# The PyTorch ecosystem — what to reach for

_PyTorch (a complete course)_

**The course finale: the libraries around PyTorch, and when to use each.**

PyTorch is a small, sharp core — tensors and autograd — surrounded by a huge ecosystem. The skill is not memorizing every library; it is knowing the three layers and reaching for the right one: the core (raw control), high-level trainers (remove boilerplate at scale), and pretrained-model hubs (skip training entirely).

---

This notebook is a practice scaffold for the **The PyTorch ecosystem — what to reach for** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q lightning
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
# Colab setup: torch is preinstalled; install the framework layers.
!pip -q install lightning transformers

# =====================================================================
# (A) PyTorch Lightning — the hand-written loop, organized away.
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

torch.manual_seed(0)

# tiny synthetic 2-class dataset (same spirit as the training-loop lesson)
N, D = 1000, 8
y = torch.randint(0, 2, (N,))
X = torch.randn(2, D)[y] * 2.0 + torch.randn(N, D)
train_dl = DataLoader(TensorDataset(X[:800], y[:800]), batch_size=32, shuffle=True)
val_dl   = DataLoader(TensorDataset(X[800:], y[800:]), batch_size=64)

class LitClassifier(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 2),                 # raw logits (no softmax!)
        )

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)     # expects logits + int labels
        self.log("train_loss", loss, prog_bar=True)
        return loss                            # Lightning calls backward()+step()

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-2)

# No epoch/batch loop, no .to(device), no zero_grad/backward/step in YOUR code.
# Flip on multi-GPU + mixed precision with flags: devices=4, precision="16-mixed".
trainer = L.Trainer(max_epochs=15, accelerator="auto", devices=1,
                    logger=False, enable_checkpointing=False)
trainer.fit(LitClassifier(), train_dl, val_dl)

# =====================================================================
# (B) Hugging Face — use a PRETRAINED model, often with no training.
# =====================================================================
from transformers import pipeline, AutoTokenizer, AutoModel

# High-level: one line loads model + tokenizer and runs the task.
clf = pipeline("sentiment-analysis")          # downloads a pretrained model
print(clf("PyTorch's ecosystem makes shipping models fast."))
# -> [{'label': 'POSITIVE', 'score': 0.99...}]

# Lower-level: load a backbone + tokenizer yourself to get embeddings.
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")
enc = tok("hello pytorch", return_tensors="pt")   # tokenizer = the model's own
with torch.no_grad():                              # inference: no graph
    out = bert(**enc)
print("sequence embedding shape:", out.last_hidden_state.shape)  # [1, seq, 768]


## Visualize the data & results

_For the same fine-tune task, how much code do you write in raw PyTorch vs. Lightning vs. Hugging Face?_

In [ ]:
import numpy as np

# Illustrative line-count breakdown for the SAME task (fine-tune + multi-GPU
# + AMP + logging), summed per stack. Components are plausible estimates;
# the point is the monotonic trend, not exact counts.
labels = ["Raw PyTorch", "PyTorch Lightning", "Hugging Face"]
#            data  model  loop  scale  logging  misc
raw   = np.array([18,   14,   40,   22,     12,    14])  # write everything
light = np.array([18,   14,   12,    0,      0,     6])  # Trainer owns loop/scale/log
hf    = np.array([ 6,    2,    0,    0,      0,     4])  # pretrained model, pipeline

vals = [int(raw.sum()), int(light.sum()), int(hf.sum())]
for name, v in zip(labels, vals):
    print(f"{name:18s} {v:3d} lines")
print("ratio raw:lightning:hf =",
      round(vals[0] / vals[2], 1), ": ",
      round(vals[1] / vals[2], 1), ": 1")
# -> Raw PyTorch        120 lines
#    PyTorch Lightning   50 lines
#    Hugging Face        12 lines


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Define a minimal LightningModule called LitNet wrapping nn.Linear(4, 2), with a training_step that computes F.cross_entropy on a (batch, labels) tuple and a configure_optimizers returning Adam(lr=1e-2). Instantiate it and print type(m).__mro__[1].__name__ to confirm it subclasses LightningModule. (lightning is auto-installed by the setup cell.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Subclass L.LightningModule and implement training_step + configure_optimizers. — _These two hooks are the minimum a LightningModule needs; the Trainer supplies the loop, device handling, and logging._
- Return the loss from training_step. — _Lightning calls backward() and optimizer.step() for you on whatever loss you return &mdash; the same five steps, organized away._

**Answer:** import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L

class LitNet(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(4, 2)        # raw logits (no softmax)
    def forward(self, x):
        return self.net(x)
    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self(x), y)   # Lightning runs backward()+step()
        return loss
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-2)

m = LitNet()
print(type(m).__mro__[1].__name__)   # LightningModule

</details>

**Problem 2.** Type this in Colab. Train the LitNet from the previous task for 2 epochs on a tiny synthetic dataset (100 samples, 4 features, 2 classes) using a Trainer. Build a DataLoader, run L.Trainer(max_epochs=2, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False).fit(...), and print "done". Use torch.manual_seed(0).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wrap the synthetic tensors in a TensorDataset + DataLoader. — _The Trainer iterates the loader; you never write the for batch loop yourself._
- Call Trainer(...).fit(model, train_dl). — _fit runs the whole loop &mdash; flipping devices=4, precision="16-mixed" would turn on multi-GPU + AMP without touching your code._

**Answer:** import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

torch.manual_seed(0)
X = torch.randn(100, 4)
y = torch.randint(0, 2, (100,))
train_dl = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

trainer = L.Trainer(max_epochs=2, accelerator="cpu", devices=1,
                    logger=False, enable_checkpointing=False)
trainer.fit(LitNet(), train_dl)
print("done")     # Lightning ran the loop for you

</details>

**Problem 3.** Type this in Colab. Use a Hugging Face pipeline with NO training. Create clf = pipeline("sentiment-analysis"), run it on the string "PyTorch's ecosystem makes shipping fast.", and print the predicted label and rounded score. (transformers is auto-installed by the setup cell.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call pipeline("sentiment-analysis"). — _One line downloads a pretrained model + its tokenizer and wires them into a ready task &mdash; zero training._
- Read result[0]["label"] and ["score"]. — _The pipeline returns a list of dicts; the label is the class and the score is its confidence._

**Answer:** from transformers import pipeline

clf = pipeline("sentiment-analysis")    # downloads a pretrained model
out = clf("PyTorch's ecosystem makes shipping fast.")
print(out[0]["label"])                  # POSITIVE
print(round(out[0]["score"], 2))        # 1.0  (high confidence)

</details>

**Problem 4.** Type this in Colab. Load a pretrained backbone with from_pretrained and inspect the embedding shape. Use AutoTokenizer and AutoModel on "bert-base-uncased", tokenize "hello pytorch" with return_tensors="pt", run it under torch.no_grad(), and print out.last_hidden_state.shape. Predict the last dimension before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pair the model's OWN tokenizer with AutoModel.from_pretrained. — _A pretrained model expects its own preprocessing; using the matching tokenizer avoids train/serve skew._
- Run inference inside torch.no_grad() and read last_hidden_state.shape. — _BERT-base outputs a 768-dim hidden state per token; the batch is 1 and seq length includes the [CLS]/[SEP] tokens._

**Answer:** import torch
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")
enc = tok("hello pytorch", return_tensors="pt")   # model's own tokenizer
with torch.no_grad():
    out = bert(**enc)
print(out.last_hidden_state.shape)   # torch.Size([1, 4, 768])
# batch 1, 4 tokens ([CLS] hello pytorch [SEP]), 768 hidden dim

</details>

**Problem 5.** Type this in Colab. Grab a pretrained torchvision model and count its parameters. Load torchvision.models.resnet18(weights="DEFAULT"), put it in eval(), and print the total parameter count with sum(p.numel() for p in model.parameters()). Then run a random (1, 3, 224, 224) image through it and print the output shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Load resnet18(weights="DEFAULT") from torchvision.models. — _Domain libraries ship ready architectures with pretrained weights, so you fine-tune instead of training from scratch._
- Run a (1, 3, 224, 224) tensor through it and read the shape. — _ImageNet ResNet-18 outputs 1000 class logits, so the shape is (1, 1000)._

**Answer:** import torch
import torchvision

model = torchvision.models.resnet18(weights="DEFAULT").eval()
print(sum(p.numel() for p in model.parameters()))   # 11689512  (~11.7M)

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = model(x)
print(out.shape)     # torch.Size([1, 1000])  -- ImageNet logits

</details>

**Problem 6.** Type this in Colab. Show that a framework does NOT change the math. Take the LitNet's training_step logic and write the equivalent RAW five-step update by hand on nn.Linear(4, 2): zero_grad, forward, cross_entropy, backward, step. Run one step on a (8, 4) batch and print the loss. Use torch.manual_seed(0).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the explicit optimizer.zero_grad() &rarr; forward &rarr; loss &rarr; backward() &rarr; step() sequence. — _These are the exact five steps Lightning's training_step hides; seeing them raw is the fundamentals the lesson insists on._
- Print loss.item(). — _Confirms the hand-written step computes the same quantity the framework would._

**Answer:** import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
model = nn.Linear(4, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

x = torch.randn(8, 4)
y = torch.randint(0, 2, (8,))

optimizer.zero_grad()              # 1
logits = model(x)                  # 2 forward
loss = F.cross_entropy(logits, y)  # 3 loss
loss.backward()                    # 4 backward
optimizer.step()                   # 5 step
print(round(loss.item(), 4))       # 0.7311

</details>

**Problem 7.** Type this in Colab. Export a model to ONNX (the deployment ring of the ecosystem) and confirm the artifact. Build nn.Linear(10, 3).eval(), call torch.onnx.export with a (1, 10) example and opset_version=17, then print os.path.exists("eco.onnx") and the file size in bytes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Call torch.onnx.export(model, example, "eco.onnx", opset_version=17). — _ONNX is the portable export format the ecosystem uses to hand a model to other runtimes (ONNX Runtime, TensorRT)._
- Verify with os.path.exists and os.path.getsize. — _Confirms the export wrote a real artifact you could ship._

**Answer:** import torch, torch.nn as nn, os

model = nn.Linear(10, 3).eval()
x = torch.randn(1, 10)
torch.onnx.export(model, x, "eco.onnx",
                  input_names=["input"], output_names=["out"],
                  opset_version=17)
print(os.path.exists("eco.onnx"))     # True
print(os.path.getsize("eco.onnx") > 0)  # True

</details>